In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  # 加载.env文件里的变量
# print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

In [ ]:
import numpy as np
import pandas as pd
import json
import io
import inspect
import requests
from langchain_core.tools import tool

@tool
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称
    注意， 中国的城市需要用对应城市的英文名称代替，例如如果要查询上海的天气，loc参数需要输入 'Shanghai'
    :return : OpenWeather API 查询即时天气的结果，具体URL请求地址为: https://api.openweathermap.org/data/2.5/weather
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    
    url=os.getenv("WEATHER_API_URL")
    
    params={
        "q":loc,
        'appid':os.getenv('WEATHER_API_KEY'),
        'units':'metric',
        'lang':'zh_cn'
    }
    
    response=requests.get(url,params=params)
    
    data=response.json()
    return json.dumps(data)

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph._internal._constants import CONF, CONFIG_KEY_RUNTIME
from langgraph.runtime import Runtime

config = {CONF: {CONFIG_KEY_RUNTIME: Runtime()}}

tools=[get_weather]
toolNode=ToolNode(tools)

pip install langchainhub


In [14]:
# from langchain_classic import hub

# prompt = hub.pull("hwchase17/react")
# prompt.pretty_print()


from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
prompt.pretty_print()


Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [17]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI

agent=create_react_agent(llm,tools,prompt)


In [18]:
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)

In [19]:
agent_executor.invoke({
    'input':"大连今天的天气"
})



> Entering new AgentExecutor chain...
Thought: 用户询问大连今天的天气，我需要使用get_weather工具查询。大连的英文名称是Dalian。  
Action: get_weather  
Action Input: Dalian  {"coord": {"lon": 121.6022, "lat": 38.9122}, "weather": [{"id": 804, "main": "Clouds", "description": "\u9634\uff0c\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 28.96, "feels_like": 34.42, "temp_min": 28.96, "temp_max": 28.96, "pressure": 1009, "humidity": 79, "sea_level": 1009, "grnd_level": 1004}, "visibility": 10000, "wind": {"speed": 3, "deg": 190}, "clouds": {"all": 93}, "dt": 1787653648, "sys": {"type": 1, "id": 9679, "country": "CN", "sunrise": 1787606151, "sunset": 1787654174}, "timezone": 28800, "id": 1814087, "name": "Dalian", "cod": 200}Thought: 我收到了大连的天气数据。根据返回的JSON结果，大连今天的天气是阴天多云，气温28.96°C，体感温度34.42°C，湿度79%，风速3米/秒。  
Final Answer: 大连今天天气为阴天多云，气温约29°C，体感温度约34°C，湿度79%，有轻微东南风，风速3米/秒。

> Finished chain.


{'input': '大连今天的天气',
 'output': '大连今天天气为阴天多云，气温约29°C，体感温度约34°C，湿度79%，有轻微东南风，风速3米/秒。'}

In [20]:
agent_executor.invoke({
    'input':"查一下今天大理，昆明和丽江哪个城市的气温最低"
})



> Entering new AgentExecutor chain...
我需要查询这三个城市的天气信息来比较气温。首先查询大理的天气。  
Action: get_weather  
Action Input: Dali  {"coord": {"lon": 100.1833, "lat": 25.7}, "weather": [{"id": 803, "main": "Clouds", "description": "\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 25.05, "feels_like": 25.26, "temp_min": 25.05, "temp_max": 25.05, "pressure": 1004, "humidity": 63, "sea_level": 1004, "grnd_level": 768}, "visibility": 10000, "wind": {"speed": 1.4, "deg": 208, "gust": 1.23}, "clouds": {"all": 58}, "dt": 1787653783, "sys": {"country": "CN", "sunrise": 1787612202, "sunset": 1787658403}, "timezone": 28800, "id": 1814093, "name": "Dali", "cod": 200}大理的气温是25.05°C。接下来查询昆明的天气。  
Action: get_weather  
Action Input: Kunming  {"coord": {"lon": 102.7183, "lat": 25.0389}, "weather": [{"id": 803, "main": "Clouds", "description": "\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 23.96, "feels_like": 23.98, "temp_min": 23.96, "temp_max": 23.96, "pressure": 1002, "hu

{'input': '查一下今天大理，昆明和丽江哪个城市的气温最低', 'output': '今天气温最低的城市是丽江，气温为21.31°C。'}